# Stage07N — A100 private inference
Chọn **Runtime → Change runtime type → A100 GPU** trước khi chạy. Upload hai file `stage07n_private_payload.tar.gz` và `run_gold_qwen_private_colab.py` lên Drive, tốt nhất vào `MyDrive/DSC2026/stage07b/`. Session L4 đang train có thể tiếp tục chạy song song. Chỉ chạy cell inference sau khi L4 tạo `REPORT.json` với `PROMOTE_TO_FULLTRAIN`.
Notebook đã nhúng runner mới độc lập; không cần upload lại file Python trước đó.


In [ ]:
import torch
assert torch.cuda.is_available(), 'Choose a GPU runtime'
gpu = torch.cuda.get_device_name(0)
print('GPU:', gpu)
assert 'A100' in gpu, f'Expected A100, got {gpu}'


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## Copy files from Drive to local A100 disk
This cell checks the exact local SHA256 hashes before copying. It searches `MyDrive/DSC2026/stage07b/` first, then the rest of `MyDrive/DSC2026/`.


In [ ]:
from pathlib import Path
import hashlib, shutil
drive_root = Path('/content/drive/MyDrive/DSC2026')
upload_dir = drive_root / 'stage07b'
expected = {
    'stage07n_private_payload.tar.gz': 'c2743b544fc885f1c1287375ba45a6a9cd8ecf9e33cd2e135f007f45d8ad77b9',
    'run_gold_qwen_private_colab.py': None,
}
def sha256(path):
    h = hashlib.sha256()
    with path.open('rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''): h.update(chunk)
    return h.hexdigest()
for name, expected_hash in expected.items():
    preferred = upload_dir / name
    if preferred.is_file():
        source = preferred
    else:
        found = list(drive_root.rglob(name))
        assert len(found) == 1, f'Expected one {name} on Drive, found {found}'
        source = found[0]
    assert expected_hash is None or sha256(source) == expected_hash, f'Wrong or incomplete upload: {source}'
    destination = Path('/content') / name
    shutil.copy2(source, destination)
    assert expected_hash is None or sha256(destination) == expected_hash
    print(f'Copied {source} -> {destination} ({destination.stat().st_size:,} bytes)')


In [ ]:
# Embed the updated standalone runner; the earlier Drive upload is accepted.
runner_source = '#!/usr/bin/env python\n"""Resume-safe private inference for the gold-supervised Stage07N checkpoint.\n\nUpload stage07n_private_payload.tar.gz and this file to /content, then run this\nscript only after the Stage07N DEV/CERT report promotes the frozen checkpoint.\nThe script imports the exact Stage07N prompt, tokenizer and score path.\n"""\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nimport os\nimport pickle\nimport sys\nimport tarfile\nimport time\nimport zipfile\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\nfrom transformers import AutoModelForCausalLM, AutoTokenizer\n\nBASE = Path("/content")\nARCHIVE = BASE / "stage07n_private_payload.tar.gz"\nPAYLOAD = BASE / "stage07n_private_payload"\nGATE = BASE / "drive/MyDrive/DSC2026/stage07n_t4_gold_qwen"\nTRAIN = Path(os.environ.get("STAGE07N_CHECKPOINT_ROOT", str(GATE)))\nOUT = TRAIN / "private_inference"\n\nPREFIX = \'<|im_start|>system\\nJudge whether the Document meets the requirements based on the Query and the Instruct provided. Note that the answer can only be "yes" or "no".<|im_end|>\\n<|im_start|>user\\n\'\nSUFFIX = \'<|im_end|>\\n<|im_start|>assistant\\n<think>\\n\\n</think>\\n\\n\'\n\n\ndef qwen_features(tok, pairs, maxlen):\n    pre = tok.encode(PREFIX, add_special_tokens=False)\n    suf = tok.encode(SUFFIX, add_special_tokens=False)\n    avail = maxlen - len(pre) - len(suf)\n    if avail < 256:\n        raise RuntimeError("maxlen too small")\n    raw = tok(pairs, padding=False, truncation=True, max_length=avail,\n              add_special_tokens=False, return_attention_mask=False)\n    return [{"input_ids": pre + row + suf} for row in raw["input_ids"]]\n\n\ndef yes_no_ids(tok):\n    yes = tok.encode("yes", add_special_tokens=False)\n    no = tok.encode("no", add_special_tokens=False)\n    if len(yes) != 1 or len(no) != 1:\n        raise RuntimeError(f"yes/no tokenizer drift: {yes}, {no}")\n    return yes[0], no[0]\n\n\ndef score_pairs(model, tok, yes_id, no_id, pairs, maxlen, amp_dtype):\n    batch = tok.pad(qwen_features(tok, pairs, maxlen), padding=True,\n                    pad_to_multiple_of=8, return_tensors="pt")\n    batch = {k: v.to("cuda", non_blocking=True) for k, v in batch.items()}\n    with torch.autocast("cuda", dtype=amp_dtype):\n        out = model.model(**batch, return_dict=True)\n        logits = model.lm_head(out.last_hidden_state[:, -1, :])\n        return (logits[:, yes_id] - logits[:, no_id]).float()\n\n\ndef zrows(x):\n    x = np.asarray(x, np.float32)\n    mu = x.mean(1, keepdims=True)\n    sd = x.std(1, keepdims=True)\n    return (x - mu) / np.where(sd < 1e-5, 1., sd)\n\n\ndef load_saved_model():\n    ck = TRAIN / "train012_model"\n    tok = AutoTokenizer.from_pretrained(ck, padding_side="left", use_fast=True)\n    kwargs = dict(torch_dtype=torch.float32, low_cpu_mem_usage=True)\n    try:\n        model = AutoModelForCausalLM.from_pretrained(ck, attn_implementation="sdpa", **kwargs)\n    except Exception:\n        model = AutoModelForCausalLM.from_pretrained(ck, **kwargs)\n    model.config.use_cache = False\n    return tok, model.to("cuda").eval()\n\n\ndef sha(path: Path) -> str:\n    h = hashlib.sha256()\n    with path.open("rb") as f:\n        for chunk in iter(lambda: f.read(1 << 20), b""):\n            h.update(chunk)\n    return h.hexdigest()\n\n\ndef load_payload():\n    if not PAYLOAD.is_dir():\n        with tarfile.open(ARCHIVE, "r:gz") as tar:\n            for item in tar.getmembers():\n                p = Path(item.name)\n                if p.is_absolute() or ".." in p.parts or p.parts[0] != PAYLOAD.name:\n                    raise RuntimeError(f"unsafe payload member {item.name}")\n            tar.extractall(BASE)\n    manifest = json.loads((PAYLOAD / "MANIFEST.json").read_text(encoding="utf-8"))\n    if manifest["private_sha256"] != "9da4e0cb84204fed924251c35744c93879556e67a440332015ea3b62f3c355bc":\n        raise RuntimeError("unexpected private population")\n    if manifest["teacher_used"] or manifest["distillation_used"]:\n        raise RuntimeError("ineligible evidence payload")\n    for name, entry in manifest["files"].items():\n        if sha(PAYLOAD / name) != entry["sha256"]:\n            raise RuntimeError(f"payload hash mismatch: {name}")\n    qids = json.loads((PAYLOAD / "qids.json").read_text(encoding="utf-8"))\n    docs = json.loads((PAYLOAD / "doc_ids.json").read_text(encoding="utf-8"))\n    short = np.load(PAYLOAD / "shortlist_top30.i4.npy")\n    prior = np.load(PAYLOAD / "ce_lr_prior.f32.npy")\n    with (PAYLOAD / "evidence_top20.pkl").open("rb") as f:\n        evidence = pickle.load(f)\n    if (len(qids), len(docs), short.shape, prior.shape, len(evidence)) != (2080, 8512, (2080, 30), (2080, 30), 2080):\n        raise RuntimeError("private payload shape drift")\n    if any(len(row) != 20 for row in evidence) or not np.isfinite(prior).all():\n        raise RuntimeError("private payload content drift")\n    return manifest, qids, docs, short, prior, evidence\n\n\ndef verify_train():\n    meta = json.loads((TRAIN / "TRAIN_META.json").read_text(encoding="utf-8"))\n    report = json.loads((GATE / "REPORT.json").read_text(encoding="utf-8"))\n    if meta.get("status") != "PASS" or meta.get("teacher_used") is not False or meta.get("distillation_used") is not False:\n        raise RuntimeError("Stage07N training provenance gate failed")\n    if report.get("decision") != "PROMOTE_TO_FULLTRAIN" or report["eligibility"].get("distillation_used") is not False:\n        raise RuntimeError(f"Stage07N DEV/CERT gate failed: {report.get(\'decision\', report.get(\'status\'))}")\n    if TRAIN == GATE and abs(float(meta["alpha"]) - float(report["alpha"])) > 1e-7:\n        raise RuntimeError("frozen alpha mismatch")\n    if TRAIN != GATE and meta.get("training_folds") != [0, 1, 2, 3, 4]:\n        raise RuntimeError("fulltrain checkpoint must contain all five gold folds")\n    if not (TRAIN / "train012_model").is_dir():\n        raise RuntimeError("Stage07N checkpoint missing")\n    return meta, report\n\n\ndef scores(model, tok, evidence, maxlen, contract):\n    OUT.mkdir(parents=True, exist_ok=True)\n    sp = OUT / "qwen_scores_top20.f32.npy"\n    dp = OUT / "qwen_done.u1.npy"\n    cp = OUT / "SCORE_CONTRACT.json"\n    if len({sp.exists(), dp.exists(), cp.exists()}) != 1:\n        raise RuntimeError("partial score cache")\n    if sp.exists():\n        if json.loads(cp.read_text(encoding="utf-8")) != contract:\n            raise RuntimeError("score cache contract mismatch")\n        result = np.lib.format.open_memmap(sp, mode="r+")\n        done = np.lib.format.open_memmap(dp, mode="r+")\n    else:\n        result = np.lib.format.open_memmap(sp, mode="w+", dtype=np.float32, shape=(len(evidence), 20))\n        done = np.lib.format.open_memmap(dp, mode="w+", dtype=np.uint8, shape=(len(evidence),))\n        done[:] = 0\n        done.flush()\n        cp.write_text(json.dumps(contract, indent=2) + "\\n", encoding="utf-8")\n    if result.shape != (len(evidence), 20) or done.shape != (len(evidence),):\n        raise RuntimeError("score cache shape drift")\n    yes_id, no_id = yes_no_ids(tok)\n    amp_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16\n    model.eval()\n    start = time.time()\n    with torch.inference_mode():\n        for qi, row in enumerate(evidence):\n            if done[qi]:\n                continue\n            s = score_pairs(model, tok, yes_id, no_id, row, maxlen, amp_dtype)\n            result[qi] = s.float().cpu().numpy()\n            done[qi] = 1\n            if (qi + 1) % 25 == 0 or qi + 1 == len(evidence):\n                result.flush()\n                done.flush()\n                elapsed = (time.time() - start) / 60\n                print(f"[private] {int(done.sum())}/{len(evidence)} elapsed_min={elapsed:.1f}", flush=True)\n    if int(done.sum()) != len(evidence) or not np.isfinite(result).all():\n        raise RuntimeError("Qwen private scores incomplete")\n    return np.asarray(result)\n\n\ndef materialize(manifest, meta, report, qids, docs, short, prior, score):\n    final = zrows(prior)\n    final[:, :20] += float(meta["alpha"]) * score\n    output = {}\n    valid = set(docs)\n    for i, qid in enumerate(qids):\n        order = np.argsort(-final[i], kind="stable")[:5]\n        answer = [docs[int(short[i, j])] for j in order]\n        if len(answer) != 5 or len(set(answer)) != 5 or any(x not in valid for x in answer):\n            raise RuntimeError(f"invalid private answer: {qid}")\n        output[qid] = {"answer": answer}\n    label = "GOLD_QWEN06B_TRAIN012_PRIVATE_K5" if TRAIN == GATE else "GOLD_QWEN06B_FULLTRAIN_PRIVATE_K5"\n    path = OUT / f"{label}.json"\n    zp = OUT / f"{label}.zip"\n    path.write_text(json.dumps(output, ensure_ascii=False, indent=2) + "\\n", encoding="utf-8")\n    info = zipfile.ZipInfo("submission.json", date_time=(2026, 9, 23, 0, 0, 0))\n    info.compress_type = zipfile.ZIP_DEFLATED\n    info.external_attr = 0o644 << 16\n    with zipfile.ZipFile(zp, "w", compresslevel=9) as z:\n        z.writestr(info, path.read_bytes())\n    with zipfile.ZipFile(zp) as z:\n        if z.namelist() != ["submission.json"] or z.read("submission.json") != path.read_bytes():\n            raise RuntimeError("ZIP payload mismatch")\n    m = {"schema": "stage07n.private_submission.v1", "status": "READY_FOR_SUBMISSION",\n         "private_sha256": manifest["private_sha256"], "queries": len(qids),\n         "teacher_used": False, "distillation_used": False,\n         "model_id": meta["model_id"], "training_folds": meta["training_folds"],\n         "dev_delta": report["dev"]["delta_recall"], "cert_delta": report["cert"]["delta_recall"],\n         "gate_report": str(GATE / "REPORT.json"),\n         "frozen_alpha": meta["alpha"], "checkpoint_dir": str(TRAIN / "train012_model"),\n         "json": str(path), "json_sha256": sha(path), "zip": str(zp), "zip_sha256": sha(zp)}\n    (OUT / "PRIVATE_SUBMISSION_REPORT.json").write_text(json.dumps(m, indent=2) + "\\n", encoding="utf-8")\n    print("READY:", zp, "SHA256", m["zip_sha256"], flush=True)\n\n\ndef main():\n    manifest, qids, docs, short, prior, evidence = load_payload()\n    meta, report = verify_train()\n    tok, model = load_saved_model()\n    maxlen = int(meta["config"]["maxlen"])\n    contract = {"schema": "stage07n.private_score_cache.v1",\n                "private_sha256": manifest["private_sha256"],\n                "evidence_sha256": manifest["files"]["evidence_top20.pkl"]["sha256"],\n                "train_meta_sha256": sha(TRAIN / "TRAIN_META.json"),\n                "maxlen": maxlen}\n    score = scores(model, tok, evidence, maxlen, contract)\n    materialize(manifest, meta, report, qids, docs, short, prior, score)\n\n\nif __name__ == "__main__":\n    main()\n'
runner_path = Path('/content/run_gold_qwen_private_colab.py')
runner_path.write_text(runner_source, encoding='utf-8')
print('Updated standalone runner:', runner_path, sha256(runner_path))


In [ ]:
%cd /content
!tar -xzf stage07n_private_payload.tar.gz
!ls -lh /content/stage07n_private_payload/MANIFEST.json /content/run_gold_qwen_private_colab.py


In [ ]:
!pip -q install 'transformers>=4.51.0' accelerate safetensors


## Gate check
Run this cell after the L4 training and DEV/CERT evaluation finish. The checkpoint and report are read from the same mounted Drive. The next cell also enforces the gate.


In [ ]:
import json
train_root = Path('/content/drive/MyDrive/DSC2026/stage07n_t4_gold_qwen')
meta_path = train_root / 'TRAIN_META.json'
report_path = train_root / 'REPORT.json'
print('TRAIN_META exists:', meta_path.exists(), 'REPORT exists:', report_path.exists())
if meta_path.exists():
    meta = json.loads(meta_path.read_text(encoding='utf-8'))
    print('train:', meta.get('status'), 'alpha:', meta.get('alpha'))
if report_path.exists():
    report = json.loads(report_path.read_text(encoding='utf-8'))
    print('decision:', report.get('decision', report.get('status')))
    print('DEV delta:', report.get('dev', {}).get('delta_recall'))
    print('CERT delta:', (report.get('cert') or {}).get('delta_recall'))


In [ ]:
%cd /content
import subprocess
with open('/content/stage07n_private_inference.log', 'a', encoding='utf-8') as log:
    process = subprocess.Popen(['python', '-u', '/content/run_gold_qwen_private_colab.py'],
                               stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                               text=True, bufsize=1)
    for line in process.stdout:
        print(line, end='')
        log.write(line)
        log.flush()
    code = process.wait()
assert code == 0, f'Private inference failed: exit {code}; inspect /content/stage07n_private_inference.log'


In [ ]:
import zipfile
result_dir = train_root / 'private_inference'
zip_path = result_dir / 'GOLD_QWEN06B_TRAIN012_PRIVATE_K5.zip'
manifest_path = result_dir / 'PRIVATE_SUBMISSION_REPORT.json'
assert zip_path.exists() and manifest_path.exists()
with zipfile.ZipFile(zip_path) as z:
    assert z.namelist() == ['submission.json']
    rows = json.loads(z.read('submission.json'))
assert len(rows) == 2080 and all(len(v['answer']) == 5 and len(set(v['answer'])) == 5 for v in rows.values())
print('READY:', zip_path)
print('SHA256:', sha256(zip_path))
print('Queries:', len(rows), 'K:', 5)
